# Sharing KV Caches Across Instances

> Two inference workers share one KV cache through Valkey — the second reads what the first wrote instead of recomputing it.

**Intermediate** · Python · ~15 min

**Who is this for:** Developers running more than one inference worker who want repeated prompts (shared system prompts, few-shot examples, common prefixes) to be computed once and reused everywhere.

## Why share KV caches?

In production you rarely run one inference worker. Multiple workers behind a load balancer often see the **same** system prompt or few-shot prefix. Without sharing, each worker recomputes identical KV tensors — wasted compute and slower first tokens.

With Valkey as a shared L2 store, whichever worker computes a chunk first writes it to Valkey; every other worker finds it and loads it instead of recomputing:

```text
Worker A: prefix -> compute KV -> store to Valkey
Worker B: same prefix -> found in Valkey -> load, skip recompute
```

This works because LMCache's key is a pure function of `(model, world_size, worker_id, chunk_hash, dtype)` — covered in [01 - Intro to KV Caching with Valkey](01-intro-to-kv-caching-with-valkey.ipynb). Two workers that hash the same prefix produce the **identical** Valkey key, so a write by one is a hit for the other.

## Prerequisites

- Completed [01 - Intro to KV Caching with Valkey](01-intro-to-kv-caching-with-valkey.ipynb)
- Docker, and the dependencies from `requirements.txt`
- No second GPU (or any GPU) needed — we simulate two workers as two independent connectors to the same Valkey

## Step 1: Start Valkey

```bash
docker compose up -d --wait
```

> ⚠️ **Security:** These examples use no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [ ]:
import subprocess
print(subprocess.run(["docker", "exec", "valkey", "valkey-cli", "ping"],
                     capture_output=True, text=True).stdout.strip())

## Step 2: Worker A computes and stores a KV chunk

Each worker is a real LMCache `ValkeyConnector`. Worker A builds the key for a
prefix, allocates the KV tensor its attention layers would produce, and stores
it to Valkey.

In [ ]:
from common import make_backend_and_connector, make_kv_tensor, build_cache_key

# The chunk_hash stands in for the hash of a tokenized prompt prefix.
PREFIX_HASH = 0xCAFE
key = build_cache_key(chunk_hash=PREFIX_HASH)

cpu_a, worker_a = await make_backend_and_connector()
mem_a, original = make_kv_tensor(cpu_a, worker_a)

await worker_a.put(key, mem_a)
print("Worker A stored:", key.to_string())
await worker_a.close()

## Step 3: Worker B reads it back without recomputing

Worker B is a completely independent connector — it never talked to Worker A.
It computes the **same** key from the **same** prefix, finds it in Valkey, and
loads the KV tensor. That load is the cache hit that lets Worker B skip the
prefill for this prefix.

In [ ]:
import torch

cpu_b, worker_b = await make_backend_and_connector()

if await worker_b.exists(key):
    got = await worker_b.get(key)
    identical = torch.equal(got.tensor.to(torch.float32), original.to(torch.float32))
    print("Worker B: cache HIT — loaded KV from Valkey, no recompute.")
    print("Bytes match what Worker A wrote:", identical)
else:
    print("Worker B: cache MISS — would compute and store instead.")

await worker_b.close()

In [ ]:
# One key in Valkey, shared by both workers.
print(subprocess.run(["docker", "exec", "valkey", "valkey-cli", "dbsize"],
                     capture_output=True, text=True).stdout.strip())

## Architecture

```text
┌─────────────┐         ┌─────────────┐
│  Worker A   │         │  Worker B   │
└──────┬──────┘         └──────┬──────┘
       │ store                 │ load
       ▼                       ▼
┌─────────────────────────────────────┐
│              Valkey (L2)             │
│   Shared KV cache — one key,         │
│   written by A, read by B            │
└─────────────────────────────────────┘
```

## What this shows — and what it doesn't

This demonstrates the **sharing mechanism**: identical inputs produce identical
Valkey keys, so a second worker reads what the first wrote.

Our demo uses one hash for the whole prompt. LMCache actually splits a prompt
into fixed-size token chunks and hashes each one, so two prompts that share only
an opening prefix can still reuse the cache for that shared part. The mechanism
you saw — same hash → same key → shared cache — is identical; only the
granularity differs.

## Where this goes next

- **[03 - Scaling to Production](03-scaling-to-production.ipynb):** cluster mode, TLS, monitoring, and wiring this under a real vLLM server.

## Teardown

```bash
docker compose down
```

---

[← 01 - Intro to KV Caching with Valkey](01-intro-to-kv-caching-with-valkey.ipynb) | [03 - Scaling to Production →](03-scaling-to-production.ipynb)